## Importing the Pandas Library

In [ ]:
import pandas as pd

housing_data = pd.read_csv("C:/Users/Degnan/Downloads/archive/housing.csv")



# The First Ten Rows of Our Data

In [ ]:
full_housing_data = housing_data.copy()
housing_data.head(10)

**Short Descriptions on Our Data**

In [ ]:
housing_data.info()

The column ocean proximity is made up of four values: Inland, Near Ocean, Near Bay, and Island. We are taking account of how many of those exists within our dataset.

In [ ]:
housing_data['ocean_proximity'].value_counts()

In [ ]:
housing_data.describe()

**Histograms of Each Numerical Attribute**

Each histogram shows how many instances fall within a given value range. The x-axis shows the value range and the y-axis shows the number of instances.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
housing_data.hist(bins=50, figsize=(20,15))
plt.show()

According to what I have understood, some of these features were capped, and the data has been preproccesed. In median income, the x-axis are actually represented of numerical dollars of tens of thousands meaning, what you see as 3 is actually about $30,000 dollars. An addition to all of this, in our median house value, the 500,000 that we see within the X-Axis may cause issues in our model, in a real-life scenario when cases like this happens, we need to communicate with the client to see whether this will be an issue or not.

**Creating the Test Set**

We will be setting aside 20% of dataset in order to check how accurate our  machine model is.

In [ ]:
import numpy as np

def split_train_test(data, test_ratio):
  shuffled_indices = np.random.permutation(len(data))

  test_size = int(len(data) * test_ratio)
  test_indices = shuffled_indices[:test_size]
  train_indices = shuffled_indices[test_size:]
  return data.iloc[train_indices], data.iloc[test_indices]

train_set, test_set = split_train_test(housing_data, 0.2)

len(train_set), len(test_set)

We need to compute hash of each instance's identifier and put that instance in the test set IF THE HASH IS LOWER THAN 20% of the maximum hash value. This ensures that the test set will remain consistent across multiple runs, because the previous coding will end up breaking at one point because it is using a different test set!

**Computationation of Hashes Concerning Each Instances Identifiers**


In [ ]:
from zlib import crc32

def test_set_check(identifier, test_ratio):
  return crc32(np.int64(identifier)) & 0xffffffff < test_ratio * 2**32

In [ ]:
def split_train_test_by_id(data, test_ratio, id_column):
  ids = data[id_column]
  in_test_set = ids.apply(lambda id_: test_set_check(id_, test_ratio))
  return data.loc[~in_test_set], data.loc[in_test_set]

Since the housing data does not have an identifer column, we will use the row index as the idenfifiers.

In [ ]:
housing_with_id = housing_data.reset_index()
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "index")

In [ ]:
display(train_set)

**Establishing the Security of Rows**

This is so that new data is appended at the end of our dataset, and no rows get deleted, we will combine the id with a feature that isn't subject to change.

In [ ]:
housing_with_id['id'] = housing_data['longitude'] * 1000 + housing_data['latitude']

train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, 'id')

The train_test_split() function from Sckit-Learn  does the same thing as split_train_test but has more additional features. These are additional features are the following:

1. random_state() - This allows us to set the set the random generator seed.
2. Passing it multiple datasets with identical number of rows will split them into the same indices.

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing_data, test_size=0.2, random_state = 42)

So far we have used purely random sampling methods for our data which is fine because our dataset is large enough for it, however if it isn't it is important to use *the stratified sampling method*, which ensures that the sampling is in representation of the population.

A *stratum* is a homogeneous subgroup that a population has been divided into. The right number of instances are **sampled** from each stratum to guarantee that the test set is in representation of the overall population.

**Median Income - An Important Attribute**

The median income attribute is an important attribute in predicting the median housing prices, so we want to ensure that the test set is in representation of our data of the various categories of incomes in the whole dataset.

Because the median income is a continuous numerical attribute, we need to create an income category attribute.

We will look at the median income histogram more closely in order for us to derive insights from it.

In [ ]:
#This code used pd.cut() in order tto create an income category attribute with five categories.
#The labels are from 1-5. Category 1 ranges from 0 to 1.5, category 2 ranges from 1.5 to 3, and so on.

housing_data['income_cat'] = pd.cut(housing_data['median_income'],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels = [1, 2, 3, 4, 5])

Most of our median income values in our previous histograms were clustered around 1.5 to 6, that is $15,000 to $60,000, and some went far beyond 6. So because of this, we must have a sufficient number of instances in our dataset for each stratum, or the estimate of the stratum's importance may become biased. So we should ensure that we do not have to many strata and each stratum is large enough.

**Representation of the Income Categories**

In [ ]:
housing_data['income_cat'].hist()

**Stratified Sampling Preparation Based Off the Income Category**

We will be using the *StratifiedShuffleSplit* class from Scikit-Learn in order to do this:

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)

for train_index, test_index in split.split(housing_data, housing_data['income_cat']):
  strat_train_set = housing_data.loc[train_index]
  strat_test_set = housing_data.loc[test_index]

**The Income Category Proportations When Using Stratified Sampling Test Dataset**

In [ ]:
strat_test_set['income_cat'].value_counts() / len(strat_test_set) #The test set generated using stratified sampling.

**The Income Category Proportations For The Overall Dataset**

In [ ]:
housing_data['income_cat'].value_counts() / len(housing_data['income_cat']) #The overall dataset concerning income categories

**The Income Category When Using The Random Sampling Test Dataset**

In [ ]:
test_set['income_cat'] = pd.cut(test_set['median_income'], #The test set when we did the purely random sampling
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels = [1, 2, 3, 4, 5])

test_set['income_cat'].hist()

test_set['income_cat'].value_counts() / len(test_set)

We have measured the income category proporation in the overall dataset, the random sampling test dataset, and also the stratified shuffled test dataset. The test set generated from the stratified sampling has income category proporations almost identical to those in the full dataset, where the ones in the purely random sampling is skewed.

**Removing the income_cat attribute in order for the data to goes back to its original state**

In [ ]:
for set_ in (strat_train_set, strat_test_set):
  set_.drop('income_cat', axis = 1, inplace = True)

During our project, we have spent much time on the generation of the test, this is a critical part of machine learning, and is often neglected. When the time comes, we will be using many of this for cross-validation. The next part of our project is data exploration.


**Data Exploration**

So far, we have take a small and small glance to have an understanding of the data we are dealing with, this time will be taking a more in-depth look at the data.

In this step, we will be exploring only the training set, so we have to put the test set aside, if the training set is large than we need to have an exploration set, so that manipulations are easier and faster. But in our case, it is much small, because of that we can work directly on the training set.

In [ ]:
housing_data = strat_train_set.copy()

Visualizing Geographical Data

Because there is geographical information such as the longitude and the latitude, we can create a scatterplot of all the districts in order to visualize the data.

In [ ]:
housing_data.plot(kind = 'scatter', x = 'longitude', y = 'latitude')

This indeed looks like California, however, so that we can visualize some patterns, we will set the alpha to 0.1. So we can see high density of data points.

A high-density residentatial area means that there is a large amount of people living in an area that is a small amount of space.

In [ ]:
housing_data.plot(kind = 'scatter', x = 'longitude', y = 'latitude', alpha = 0.1)

We want to make these patterns stand out, however what we can see from here is the high-density area such as Los Angeles, San Diego, and the Central Valley mainly in Sacramento and Fresno.

We will make this visualization more appealing.

**Protuded Visualization**

In [ ]:
housing_data.plot(kind = 'scatter',
                  x = 'longitude',
                  y = 'latitude',
                  alpha = 0.4,
                  s = housing_data['population']/100,
                  label = 'population',
                  figsize = (10, 7),
                  c = 'median_house_value',
                  cmap = plt.get_cmap('jet'),
                  colorbar = True,
                  )
plt.legend()

The radius of each of the circle represents the population of that district. The color represents the price. From blue which means low prices to red meaning high price, we have used a predefined color mapping.

The image has given us how the pricing of the houses is in correlation to location and the population density, for example the houses the are closer to the ocean are at a much high price.

**Searching For Correlations**

Because the dataset isn't too large, we can use the standard correlation known as Pearson's r between every pair of attributes using the correlation method.

We have used the scatter_matrix method from pandas in order to plot every attribute against other attributes. In total, we would've receive 121 plots, but we focused on more promising attributes that appears to be in correlation with the median_house.

In [ ]:
from pandas.plotting import scatter_matrix

attributes = ['median_house_value', 'median_income', 'total_rooms', 'housing_median_age']

scatter_matrix(housing_data[attributes], figsize = (12, 8))


In [ ]:
housing_data.plot(kind = 'scatter', x = 'median_income', y = 'median_house_value', alpha = 0.1)

The median_income is the most promising in predicting the median_house value. In our visualization, the correlation is very strong because it shows an upward trend and the data points aren't too dispersed.

We can also see the price cap of 500,000 in the form of a horizontal line, we see another at 450,000 and yet another at 350,000. These are data quirks that may need to be removed so that it doesn't have an interference with our algorithm.

**Experimenting with Attribute Combinations**

Before we prepare our data for our machine learning model, we can create new features by combining existing ones, this is so that we can improve our model's performance. The total number of rooms in a district isn't useful if you don't know the number of households.

What we really want is the number of rooms per household. We will also take a look at the population per household as well and combine it.

In [ ]:
housing_data['rooms_per_household'] = housing_data['total_rooms']/housing_data['households']

housing_data['bedrooms_per_room'] = housing_data['total_bedrooms']/housing_data['total_rooms']

housing_data['population_per_household'] = housing_data['population']/housing_data['households']

In [ ]:
housing_data = housing_data.drop(columns = ['ocean_proximity'])

corr_matrix = housing_data.corr()
corr_matrix['median_house_value'].sort_values(ascending = False)

**Prepare the Data for Machine Learning Algorithm**

Before we prepare our data, instead of doing it manually, we will do them as functions for the purpose of efficiency and reusability.

Additionally, we don't want to do the same transformations on both the predictors and the target values, because of this we will do a seperation between the two.

In [ ]:
housing_data = strat_train_set.drop("median_house_value", axis = 1)
housing_labels = strat_train_set['median_house_value'].copy()

**Data Cleaning**

Machine learning algorithms usually cannot work with missing features. The attribute of total_bedrooms was known to have been missing values, in order for us to counter this, we can use some one pandas functions such as dropna(), drop(), *fillna()*.

In [ ]:
# Three options for handling missing values (only option 3 is applied here):
# Option 1: housing_data.dropna(subset=['total_bedrooms'])   -> drop the districts with missing values
# Option 2: housing_data.drop('total_bedrooms', axis=1)       -> drop the whole attribute
# Option 3: fill the missing values with the median
median = housing_data['total_bedrooms'].median()
housing_data['total_bedrooms'] = housing_data['total_bedrooms'].fillna(median)

Another way to handle way we can handle missing value is by using *SimpleImputer* class from Scikit-learn.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy = 'median')

In [ ]:
# We can now fit the imputer instance to the training data set (after removing the text column below).

Because *ocean_proximity* is a categorical value, and the median value can only be computed on numerical attributes, we will create a copy of the data without that attribute.

In [ ]:
housing_num = housing_data.drop('ocean_proximity', axis = 1)

In [ ]:
imputer.fit(housing_num)

After the *imputer* has calculated the median of each attributes, it will store the results within its statistics variable. In our data, only the total_bedrooms had missing values, however it is possible that in new data that there may be missing values, so we will use the *imputer* on all the numerical variables.

In [ ]:
imputer.statistics_

In [ ]:
housing_num.median().values

We can use the "trained" *imputer* to transform the training set by replacing the missing values with the learned medians.

In [ ]:
X = imputer.transform(housing_num)

We ended up receiving a plain Numpy array of transformed features then placed it into a pandas DataFrame.

In [ ]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns,
                          index=housing_num.index)

**Handling Text and Categorical Attributes**

In our dataset, we've only dealth with the numerical attributes. We setted aside the only categorical attribute, however now will look further into this attribute. The first thing we will do is look at the first 10 instances.

In [ ]:
housing_categorical = housing_data[['ocean_proximity']]
housing_categorical.head(10)

Most machine learning algorithms prefer to work with numerical values rather then categorical ones. Because *'ocean_proximity'* is a categorical value we will convert this to numerical values using encoders.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
ordinal_encoder = OrdinalEncoder()
housing_categorical_encoded = ordinal_encoder.fit_transform(housing_categorical)
housing_categorical_encoded[:10]

By using the *categories_* instance variable, we are able to receive a list concerning the identification array of the different categories for each categorical attribute.

In [ ]:
ordinal_encoder.categories_

Within machine learning algorithms, one of the possible issues that may happen is that our algorithm may think that values(categories in this case) that are nearby are similar. In some cases, this may be fine, however in our case this is a problem. In *ocean_proximity*, categories 0 and 4 are more similar than 0 and 1, and this values or categories are not near by.

The solution to an issue like this is by using *One Hot Encoder*, this will convert categorical to one-hot vectors.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
categorical_encoder = OneHotEncoder()
housing_categorical_one_hot = categorical_encoder.fit_transform(housing_categorical)
housing_categorical_one_hot

The output is a SciPy *sparse matrix* rather than a NumPy array. After one-hot encoding we get one column per category (5 columns here), and each row has a single 1 with 0s everywhere else. When there are thousands of categories, storing all those zeros would waste memory, so a sparse matrix stores only the locations of the nonzero values. We can convert it to a dense NumPy array with the *toarray()* method.

In [ ]:
housing_categorical_one_hot.toarray()

**Custom Transformers**

In Scikit-Learn, a *transformer* is any object with `fit()` and `transform()` methods that changes the data, such as `SimpleImputer` or `StandardScaler`. (This is unrelated to the Transformer neural network architecture used in language models.) Scikit-Learn provides many transformers, but sometimes we need our own for custom cleanup operations or for combining specific attributes. We will write a small transformer class that adds the combined attributes we created earlier.

Inheriting from `TransformerMixin` gives us `fit_transform()` for free, and inheriting from `BaseEstimator` gives us `get_params()` and `set_params()`, which are needed for hyperparameter tuning.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
  def __init__(self, add_bedrooms_per_room = True):
    self.add_bedrooms_per_room = add_bedrooms_per_room
  def fit(self, X, y = None):
    return self
  def transform(self, X):
    rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
    population_per_household = X[:, population_ix] / X[:, households_ix]
    if self.add_bedrooms_per_room:
      bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
      return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
    else:
      return np.c_[X, rooms_per_household, population_per_household]

attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing_data.values)

Variables that can be configured are known as hyperparameters. Hyperparameters control the learning process, model structure, function, and performance. These hyperparameters are set by the user before the training has begun and are external to the model, causing it not be changed during training.

In our case, our transformer has only one hyperparameter, *add_bedrooms_per_room*, which defaults to *True* (above we set it to *False* to try it out), by adding this hyperparameter, it will help to know whether this attribute helps our algorithm or not.

**Feature Scaling** - Machine learning algorithms don't perform well when the input numerical attributes have different scales. There are two common ways to get all the attributes to have the same scale and that is by using min-max scaling and standardization.

**Standardization** - Different from min-max scaling, standardization does not bound values to a specific range and it much less effected by outliers. We will be writing down the code to do this in the form of **transformation pipelines**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler()), ])

housing_num_tr = num_pipeline.fit_transform(housing_num)

### Numerical Preprocessing Pipeline

A `Pipeline` chains the preprocessing steps for the **numerical features** so they always run in the same order:

1. **`imputer`**: `SimpleImputer(strategy='median')` fills missing values with each column's median, which is robust to outliers.
2. **`attribs_adder`**: `CombinedAttributesAdder()` is a custom transformer that adds engineered features, such as rooms per household, population per household, and bedrooms per room.
3. **`std_scaler`**: `StandardScaler()` standardizes every feature to mean 0 and standard deviation 1.

`fit_transform(housing_num)` fits each step on the data and transforms it before passing the result to the next step. It returns a NumPy array.

**Notes:**
- Every step except the last must be a transformer, meaning it has `fit()` and `transform()`.
- The pipeline's methods mirror the final step's methods.
- Use `fit_transform()` on the training data and only `transform()` on the test or new data, to avoid data leakage.
- You can access a step by name, e.g. `num_pipeline.named_steps['imputer']`, or tune it with grid-search parameters like `imputer__strategy`.


In [ ]:
from sklearn.compose import ColumnTransformer

num_attribs = list(housing_num)
cat_attribs = ['ocean_proximity']

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs),
])

housing_prepared_data = full_pipeline.fit_transform(housing_data)

### Full Preprocessing with `ColumnTransformer`

`ColumnTransformer` applies **different transformations to different columns** and concatenates the results into a single output.

- `num_attribs = list(housing_num)`: the names of all numerical columns.
- `cat_attribs = ['ocean_proximity']`: the categorical column.

Each entry is a `(name, transformer, columns)` tuple:

| Name | Transformer | Columns | What it does |
|------|-------------|---------|--------------|
| `num` | `num_pipeline` | `num_attribs` | Impute → add attributes → scale |
| `cat` | `OneHotEncoder()` | `cat_attribs` | One binary column per category |

`fit_transform(housing_data)` fits and transforms each column group, then joins the results side by side, ready for model training.

**Notes:**
- Columns that aren't listed are **dropped** by default (`remainder='drop'`). Use `remainder='passthrough'` to keep them.
- The output can be **sparse or dense**, depending on how many of its values are zero (`sparse_threshold=0.3`).
- `OneHotEncoder` raises an error on unseen categories. Use `handle_unknown='ignore'` if new data might contain them.
- `housing_data` should contain the features only, without the label `median_house_value`.


## Selecting and training a model

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(housing_prepared_data, housing_labels)

In [ ]:
some_data = housing_data.iloc[:5]
some_labels = housing_labels.iloc[:5]
some_data_prepared = full_pipeline.transform(some_data)
print("Predictions:", lin_reg.predict(some_data_prepared))
print("Labels:", list(some_labels))